In [ ]:
%pip install semantic-link-labs

In [ ]:
import pandas as pd
from typing import Optional
from uuid import UUID
from sempy._utils._log import log
from sempy_labs._helper_functions import (
    resolve_workspace_name_and_id,
    _base_api,
    _create_dataframe,
    _build_url,
)

@log
def find_semantic_models_with_personal_connections(
    workspace: Optional[str | UUID] = None,
) -> pd.DataFrame:
    """
    Identifies semantic models that use a personal (non-gateway) connection.

    A personal connection is indicated when a datasource has no associated
    gateway (gatewayId is null). These are cloud-only direct connections
    tied to the individual who published the report, which can cause refresh
    failures if that person's credentials expire or they leave the org.

    Parameters
    ----------
    workspace : str | uuid.UUID, default=None
        The Fabric workspace name or ID to scope the search.
        If None, scans ALL workspaces (requires admin permissions).

    Returns
    -------
    pandas.DataFrame
        A dataframe of semantic models using personal connections.
        Columns: 'Workspace Id', 'Workspace Name', 'Dataset Id',
                 'Dataset Name', 'Datasource Type', 'Connection Details'
    """

    columns = {
        "Workspace Id":        "string",
        "Workspace Name":      "string",
        "Dataset Id":          "string",
        "Dataset Name":        "string",
        "Datasource Type":     "string",
        "Connection Details":  "string",
    }
    df = _create_dataframe(columns=columns)
    rows = []

    # ── Step 1: Collect workspaces to scan ──────────────────────────────────
    if workspace is not None:
        (ws_name, ws_id) = resolve_workspace_name_and_id(workspace)
        workspaces = [{"id": str(ws_id), "name": ws_name}]
    else:
        # Admin endpoint — lists all workspaces in the org
        print("No workspace specified — scanning all workspaces (admin required)...")
        ws_url = _build_url(
            "/v1.0/myorg/admin/groups",
            {"$top": "5000", "$filter": "type eq 'Workspace' and state eq 'Active'"},
        )
        ws_resp = _base_api(request=ws_url, client="fabric_sp").json()
        workspaces = [
            {"id": g["id"], "name": g["name"]}
            for g in ws_resp.get("value", [])
        ]
        print(f"Found {len(workspaces)} active workspaces.")

    # ── Step 2 & 3: For each workspace, get datasets then their datasources ──
    for ws in workspaces:
        ws_id   = ws["id"]
        ws_name = ws["name"]

        # List all datasets in this workspace
        try:
            ds_resp = _base_api(
                request=f"/v1.0/myorg/groups/{ws_id}/datasets",
                client="fabric_sp",
            ).json()
        except Exception as e:
            print(e)
            continue  # skip workspaces we can't read

        for dataset in ds_resp.get("value", []):
            dataset_id   = dataset.get("id")
            dataset_name = dataset.get("name")

            # Get datasources for this dataset
            try:
                src_resp = _base_api(
                    request=f"/v1.0/myorg/groups/{ws_id}/datasets/{dataset_id}/datasources",
                    client="fabric_sp",
                ).json()
                print(src_resp)
            except Exception as e:
                print(e)
                continue

            for src in src_resp.get("value", []):
                gateway_id = src.get("gatewayId")

                # ── Filter: null gatewayId = personal connection ──
                if not gateway_id:
                    conn_details = src.get("connectionDetails", {})
                    rows.append({
                        "Workspace Id":       ws_id,
                        "Workspace Name":     ws_name,
                        "Dataset Id":         dataset_id,
                        "Dataset Name":       dataset_name,
                        "Datasource Type":    src.get("datasourceType", "Unknown"),
                        "Connection Details": str(conn_details),
                    })

    if rows:
        df = pd.DataFrame(rows).drop_duplicates(
            subset=["Dataset Id", "Datasource Type", "Connection Details"]
        )

    print(f"Found {len(df)} semantic model datasource(s) using personal connections.")
    return df




In [ ]:
# ── Run it ───────────────────────────────────────────────────────────────────
# Scope to one workspace:
results = find_semantic_models_with_personal_connections(workspace="Dev")

# Or scan the whole tenant (needs admin):
# results = find_semantic_models_with_personal_connections()

display(results)

In [ ]:
import pandas as pd
from typing import Optional
from uuid import UUID
from sempy._utils._log import log
from sempy_labs._helper_functions import (
    resolve_workspace_name_and_id,
    _base_api,
    _create_dataframe,
    _build_url,
)

# connectivityType values that indicate a personal (non-shareable) connection
_PERSONAL_CONNECTIVITY_TYPES = {"PersonalCloud", "OnPremisesGatewayPersonal"}


@log
def find_semantic_models_with_personal_connections(
    workspace: Optional[str | UUID] = None,
) -> pd.DataFrame:
    """
    Identifies semantic models that use a personal connection.

    Uses the Fabric List Item Connections API to inspect each semantic model's
    connection bindings and flags any with a ``connectivityType`` of
    ``PersonalCloud`` or ``OnPremisesGatewayPersonal``. Personal connections
    are tied to an individual user's credentials and cannot be shared, which
    can cause refresh failures when that user's token expires or they leave
    the organisation.

    This is a wrapper function for the following API:
    `Items - List Item Connections <https://learn.microsoft.com/rest/api/fabric/core/items/list-item-connections>`_.

    Service Principal Authentication is supported (see `here <https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Service%20Principal.ipynb>`_ for examples).

    Parameters
    ----------
    workspace : str | uuid.UUID, default=None
        The Fabric workspace name or ID to scope the search.
        If None, scans ALL workspaces visible to the caller. Tenant-wide
        scans require the Power BI Service Administrator role or a service
        principal with Tenant.Read.All.

    Returns
    -------
    pandas.DataFrame
        A dataframe listing semantic models that have at least one personal
        connection binding.
        Columns: 'Workspace Id', 'Workspace Name', 'Semantic Model Id',
                 'Semantic Model Name', 'Connection Id', 'Connectivity Type',
                 'Datasource Type', 'Connection Path'.
    """

    columns = {
        "Workspace Id":          "string",
        "Workspace Name":        "string",
        "Semantic Model Id":     "string",
        "Semantic Model Name":   "string",
        "Connection Id":         "string",
        "Connectivity Type":     "string",
        "Datasource Type":       "string",
        "Connection Path":       "string",
    }
    df = _create_dataframe(columns=columns)
    rows = []

    # ── Step 1: Resolve workspaces to scan ──────────────────────────────────
    if workspace is not None:
        (ws_name, ws_id) = resolve_workspace_name_and_id(workspace)
        workspaces = [{"id": str(ws_id), "name": ws_name}]
    else:
        print("No workspace specified — scanning all workspaces (admin role required)...")
        ws_url = _build_url(
            "/v1.0/myorg/admin/groups",
            {"$top": "5000", "$filter": "type eq 'Workspace' and state eq 'Active'"},
        )
        ws_resp = _base_api(request=ws_url, client="fabric_sp").json()
        workspaces = [
            {"id": g["id"], "name": g["name"]}
            for g in ws_resp.get("value", [])
        ]
        print(f"Found {len(workspaces)} active workspaces.")

    # ── Step 2: List semantic models in each workspace ───────────────────────
    for ws in workspaces:
        ws_id   = ws["id"]
        ws_name = ws["name"]

        try:
            items_resp = _base_api(
                request=_build_url(
                    f"/v1/workspaces/{ws_id}/items",
                    {"type": "SemanticModel"},
                ),
                client="fabric_sp",
            ).json()
        except Exception:
            continue

        semantic_models = items_resp.get("value", [])

        # ── Step 3: Check connection bindings for each model ─────────────────
        for model in semantic_models:
            model_id   = model.get("id")
            model_name = model.get("displayName")

            try:
                conn_resp = _base_api(
                    request=f"/v1/workspaces/{ws_id}/items/{model_id}/connections",
                    client="fabric_sp",
                ).json()
            except Exception:
                continue

            for conn in conn_resp.get("value", []):
                connectivity_type = conn.get("connectivityType", "")

                # ── Filter: personal connectivity types only ──────────────
                if connectivity_type in _PERSONAL_CONNECTIVITY_TYPES:
                    conn_details = conn.get("connectionDetails", {})
                    rows.append({
                        "Workspace Id":        ws_id,
                        "Workspace Name":      ws_name,
                        "Semantic Model Id":   model_id,
                        "Semantic Model Name": model_name,
                        "Connection Id":       conn.get("id", ""),
                        "Connectivity Type":   connectivity_type,
                        "Datasource Type":     conn_details.get("type", ""),
                        "Connection Path":     conn_details.get("path", ""),
                    })

    if rows:
        df = pd.DataFrame(rows)

    print(f"Found {len(df)} personal connection binding(s) across semantic models.")
    return df




In [ ]:
# ── Run it ───────────────────────────────────────────────────────────────────
# Scope to one workspace:
results = find_semantic_models_with_personal_connections(workspace="Dev")

# Or scan the whole tenant (needs admin):
# results = find_semantic_models_with_personal_connections()

display(results)